# FD004 Paper 2: P2-E1 (30-cycle) vs P2-E8 (60-cycle) point models, clean rerun

The original P2-E1..E8 runs stopped early on the 49 IID validation engines and then reported RMSE on the same engines.
This rerun keeps the Paper 2 point architecture (18-dim linear projection, LSTM(64), Dense(1), MSE, Adam 1e-3, batch 64)
and changes only how training stops, using the Paper 1 rule written down before evaluation:

- internal validation = 20% of the 200 IID training engines (split seed 42); the 49 validation engines are never used to train or stop;
- up to 150 epochs, early stopping on internal-validation MSE, patience 15, not before epoch 30, best weights restored;
- converged if best internal-validation MSE ≤ 0.60 × the MSE of predicting the training mean;
- seeds 42, 123, 456, 789, 1000, all reported.

Evaluation: all 49 IID validation engines, **matched endpoints only** (endpoint cycle ≥ 60, which both models can score).

In [1]:
import os, sys, json, time, random, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd
SMOKE_TEST = os.environ.get("FD004_SMOKE") == "1"
def _find_root():
    for c in [os.environ.get("FD004_ROOT", ""), "/content/drive/MyDrive/FD004_Research", str(Path.home() / "Downloads/FD004_Research")]:
        if c and Path(c, "CMAPSSData/train_FD004.txt").exists():
            return Path(c)
    raise FileNotFoundError("Set FD004_ROOT")
ROOT = _find_root()
OUT = Path(os.environ.get("FD004_OUT") or ROOT / ("FD004_P2E1_vs_P2E8_Results" + ("_SMOKE" if SMOKE_TEST else "")))
(OUT / "models").mkdir(parents=True, exist_ok=True); (OUT / "histories").mkdir(exist_ok=True)
PROTOCOL = {"experiment": "P2-E1 (30) vs P2-E8 (60) point LSTM, Paper 1 training rule, matched IID endpoints",
            "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "seeds": [42] if SMOKE_TEST else [42, 123, 456, 789, 1000], "windows": [30, 60],
            "model": "Dense(18, no bias) -> LSTM(64) -> Dense(1), MSE, Adam 1e-3, batch 64",
            "internal_validation": "20% of 200 IID training engines, split seed 42",
            "training_rule": {"max_epochs": 3 if SMOKE_TEST else 150, "patience": 15, "start_from_epoch": 0 if SMOKE_TEST else 30},
            "convergence_check": "best val MSE <= 0.60 x MSE of training-mean predictor",
            "evaluation": "49 IID validation engines, endpoints with cycle >= 60 (matched)",
            "life_stage": {"Early": "RUL > 80", "Mid": "40 < RUL <= 80", "Late": "RUL <= 40"}}
pp = OUT / "protocol.json"
if pp.exists() and not SMOKE_TEST:
    old = json.loads(pp.read_text()); assert {k for k in PROTOCOL if k != "created_utc" and old.get(k) != PROTOCOL[k]} == set()
else:
    pp.write_text(json.dumps(PROTOCOL, indent=2))
print("Output:", OUT)

Output: <FD004_ROOT>/FD004_P2E1_vs_P2E8_Results


In [2]:
from sklearn.model_selection import train_test_split
b = np.load(ROOT / "artifacts/FD004_baseline_tensors.npz"); t = np.load(ROOT / "artifacts/P2_E8_60cycle_tensors.npz")
def eng(y):
    i = np.ones(len(y), int); i[1:] = 1 + np.cumsum(np.diff(y) > 0); return i
def onehot(X):
    return np.concatenate([X[:, :, :14], np.eye(6, dtype=np.float32)[X[:, :, 14].astype(int)]], -1).astype(np.float32)
data = {}
# 30-cycle: engine index by order of appearance; endpoint cycle = position within engine + 30
id_tr30, id_v30 = eng(b["y_train"]), eng(b["y_val"])
cyc_v30 = np.concatenate([np.arange(c) + 30 for c in np.bincount(id_v30)[1:]])
data[30] = dict(Xtr=onehot(b["X_train"]), ytr=b["y_train"].astype(np.float32), idtr=id_tr30,
                Xv=onehot(b["X_val"]), yv=b["y_val"].astype(np.float32), idv=id_v30, cyc=cyc_v30)
id_tr60 = 1 + pd.Index(pd.unique(t["train_endpoint_unit"])).get_indexer(t["train_endpoint_unit"])
id_v60 = 1 + pd.Index(pd.unique(t["val_endpoint_unit"])).get_indexer(t["val_endpoint_unit"])
data[60] = dict(Xtr=onehot(t["X_train"]), ytr=t["y_train"].astype(np.float32), idtr=id_tr60,
                Xv=onehot(t["X_val"]), yv=t["y_val"].astype(np.float32), idv=id_v60, cyc=t["val_endpoint_cycle"])
assert np.all(np.bincount(id_v30)[1:] - np.bincount(id_v60)[1:] == 30)
fit_e, ival_e = train_test_split(np.arange(1, 201), test_size=0.20, random_state=42)
for w, d in data.items():
    d["mf"], d["mi"] = np.isin(d["idtr"], fit_e), np.isin(d["idtr"], ival_e)
    d["match"] = d["cyc"] >= 60
    print(w, "fit", d["mf"].sum(), "ival", d["mi"].sum(), "eval matched", d["match"].sum())
assert data[30]["match"].sum() == data[60]["match"].sum() == len(data[60]["yv"])
assert np.array_equal(data[30]["yv"][data[30]["match"]], data[60]["yv"]) and np.array_equal(data[30]["idv"][data[30]["match"]], data[60]["idv"])

30 fit 34983 ival 8713 eval matched 8862
60 fit 30183 ival 7513 eval matched 8862


In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM
def build(w):
    inp = Input(shape=(w, 20)); h = Dense(18, use_bias=False)(inp); h = LSTM(64)(h)
    m = tf.keras.Model(inp, Dense(1)(h)); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse"); return m
def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); os.environ["PYTHONHASHSEED"] = str(s)
tr = PROTOCOL["training_rule"]; log, preds = [], []
for w, d in data.items():
    Xf, yf, Xi, yi = d["Xtr"][d["mf"]], d["ytr"][d["mf"]], d["Xtr"][d["mi"]], d["ytr"][d["mi"]]
    const = float(np.mean((yi - yf.mean()) ** 2))
    for seed in PROTOCOL["seeds"]:
        tf.keras.backend.clear_session(); set_seed(seed); m = build(w)
        cb = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=tr["patience"], start_from_epoch=tr["start_from_epoch"], restore_best_weights=True)
        h = m.fit(Xf, yf, validation_data=(Xi, yi), epochs=tr["max_epochs"], batch_size=64, shuffle=True, callbacks=[cb], verbose=2)
        h = pd.DataFrame(h.history); h.insert(0, "epoch", np.arange(1, len(h) + 1)); h.to_csv(OUT / "histories" / f"W{w}_seed{seed}.csv", index=False)
        m.save(OUT / "models" / f"P2E{'1' if w == 30 else '8'}_W{w}_seed{seed}.keras")
        best = float(h.val_loss.min())
        log.append({"window": w, "seed": seed, "epochs_run": len(h), "best_epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
                    "best_val_mse": best, "constant_val_mse": const, "ratio": best / const, "converged": best <= 0.60 * const})
        p = m.predict(d["Xv"][d["match"]], batch_size=1024, verbose=0).reshape(-1)
        preds.append(pd.DataFrame({"window": w, "seed": seed, "engine": d["idv"][d["match"]], "cycle": d["cyc"][d["match"]],
                                   "true_RUL": d["yv"][d["match"]], "pred": p}))
        print(log[-1])
log = pd.DataFrame(log); log.to_csv(OUT / "convergence_check.csv", index=False)
pred = pd.concat(preds, ignore_index=True); pred["error"] = pred.pred - pred.true_RUL
pred["stage"] = np.where(pred.true_RUL > 80, "Early", np.where(pred.true_RUL > 40, "Mid", "Late"))
pred.to_csv(OUT / "matched_predictions.csv", index=False)
print(log.round(3).to_string(index=False))

Epoch 1/150


547/547 - 6s - 10ms/step - loss: 5788.9697 - val_loss: 3839.1145


Epoch 2/150


547/547 - 4s - 8ms/step - loss: 2905.5640 - val_loss: 2239.1536


Epoch 3/150


...

472/472 - 13s - 27ms/step - loss: 52.5303 - val_loss: 254.7454


Epoch 46/150


472/472 - 12s - 26ms/step - loss: 44.6561 - val_loss: 249.5154


{'window': 60, 'seed': 1000, 'epochs_run': 46, 'best_epoch': 16, 'best_val_mse': 182.3473663330078, 'constant_val_mse': 1764.5155029296875, 'ratio': 0.10334132289019282, 'converged': True}


 window  seed  epochs_run  best_epoch  best_val_mse  constant_val_mse  ratio  converged
     30    42          52          18       268.008          1718.057  0.156       True
     30   123          47          22       291.295          1718.057  0.170       True
     30   456          46          13       264.518          1718.057  0.154       True
     30   789          46          16       269.354          1718.057  0.157       True
     30  1000          46          14       299.986          1718.057  0.175       True
     60    42          50          11       187.671          1764.516  0.106       True
     60   123          46          13       170.620          1764.516  0.097       True
     60   456          51          13       195.576          1764.516  0.111       True
     60   789          48          16       181.869          1764.516  0.103       True
     60  1000          46          16       182.347          1764.516  0.103       True


In [4]:
rmse = lambda e: float(np.sqrt(np.mean(e ** 2)))
rows = []
for (w, seed), g in pred.groupby(["window", "seed"]):
    r = {"window": w, "seed": seed, "N": len(g), "RMSE": rmse(g.error), "MAE": float(g.error.abs().mean()), "bias": float(g.error.mean())}
    for s_, gs in g.groupby("stage"):
        r[f"{s_}_RMSE"] = rmse(gs.error); r[f"{s_}_bias"] = float(gs.error.mean())
    rows.append(r)
res = pd.DataFrame(rows); res.to_csv(OUT / "matched_metrics_by_seed.csv", index=False)
summ = res.groupby("window").agg(["mean", "std"]).drop(columns=["seed"], level=0); summ.to_csv(OUT / "matched_metrics_five_seed.csv")
print(summ.round(3).T.to_string())
w30, w60 = res[res.window == 30].set_index("seed"), res[res.window == 60].set_index("seed")
diff = (w60[["RMSE", "Mid_RMSE", "Early_RMSE", "Late_RMSE"]] - w30[["RMSE", "Mid_RMSE", "Early_RMSE", "Late_RMSE"]])
diff.to_csv(OUT / "paired_difference_60_minus_30_by_seed.csv"); print(diff.round(3).to_string()); print(diff.agg(["mean", "std"]).round(3))
# seed-42 engine-cluster bootstrap of the 60 - 30 RMSE difference (global and mid-life)
a = pred[(pred.seed == 42) & (pred.window == 30)].reset_index(drop=True); c = pred[(pred.seed == 42) & (pred.window == 60)].reset_index(drop=True)
assert np.array_equal(a.engine.values, c.engine.values) and np.array_equal(a.cycle.values, c.cycle.values)
engs = np.unique(a.engine); rng = np.random.default_rng(42); out = []
by = {e: np.where(a.engine.values == e)[0] for e in engs}
for _ in range(10000):
    idx = np.concatenate([by[e] for e in rng.choice(engs, len(engs))])
    mid = idx[a.stage.values[idx] == "Mid"]
    out.append((rmse(c.error.values[idx]) - rmse(a.error.values[idx]), rmse(c.error.values[mid]) - rmse(a.error.values[mid])))
out = np.array(out)
boot = pd.DataFrame({"stat": ["RMSE 60-30", "Mid RMSE 60-30"],
                     "point": [rmse(c.error) - rmse(a.error), rmse(c.error[c.stage == "Mid"]) - rmse(a.error[a.stage == "Mid"])],
                     "ci_low": np.percentile(out, 2.5, axis=0), "ci_high": np.percentile(out, 97.5, axis=0)})
boot.to_csv(OUT / "bootstrap_seed42.csv", index=False); print(boot.round(3).to_string(index=False))
manifest = {str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob("*")) if p.is_file()}
(OUT / "manifest_sha256.json").write_text(json.dumps(manifest, indent=2)); print("Done")

window                 30        60
N          mean  8862.000  8862.000
           std      0.000     0.000
RMSE       mean    22.689    19.176
           std      0.690     0.386
MAE        mean    15.565    13.414
           std      0.401     0.461
bias       mean    -1.011    -1.655
           std      1.237     1.243
Early_RMSE mean    22.262    19.863
           std      1.198     0.697
Early_bias mean    -8.053    -7.677
           std      1.144     1.559
Late_RMSE  mean    13.788    10.329
           std      1.178     0.622
Late_bias  mean     4.778     3.175
           std      1.072     1.047
Mid_RMSE   mean    29.929    23.837
           std      0.744     0.884
Mid_bias   mean    10.778     8.552
           std      1.788     1.440
       RMSE  Mid_RMSE  Early_RMSE  Late_RMSE
seed                                        
42   -3.938    -5.074      -3.739     -3.230
123  -2.708    -7.764       0.088     -4.324
456  -3.987    -5.717      -3.738     -2.232
789  -2.774    -5.5

          stat  point  ci_low  ci_high
    RMSE 60-30 -3.938  -6.212   -1.779
Mid RMSE 60-30 -5.074  -8.231   -1.847
Done
